# Детекция ботов по событиям внутри суточного окна

Задача: по событиям куки внутри окна `[window_start_ts, window_end_ts)` предсказать `score` —
вероятность того, что кука принадлежит сервису автоматизированного сбора данных.
Метрика — `Precision @ Recall >= 0.70`, считается по кукам, код взят из приложенного `metric.py`.

Ноутбук выполняется сверху вниз и создаёт `submission.csv`. Полный прогон — около 20 минут,
основное время уходит на валидацию и на проверку признаков контекста.

---

## Описание подхода

**Признаки.** Готовых нет, всё собирается из `events.csv.gz`. Одиннадцать групп, от простых
счётчиков до контекста по корпусу. Ключевая идея: робот дёшево наращивает объём и дорого имитирует
нерегулярность, поэтому вес отдан не счётчикам, а форме распределения интервалов, неравномерности
обхода каталога и пиковой интенсивности.

**Валидация.** Тест лежит позже трейна, поэтому валидация временна́я: пять расширяющихся окон,
обучение всегда на всех данных строго раньше валидационного среза.

**Модель.** `HistGradientBoostingClassifier` из scikit-learn, 24 модели на бутстрэп-выборках,
свёртка — **медиана рангов**. Внешних зависимостей и API нет.

**Результат.** `P@R0.7` = 0.827 на валидации, 0.839 на скрытом тесте при константе 0.094 и
baseline из quickstart 0.099.

**Использованные библиотеки:** numpy, pandas, scikit-learn, scipy. Версии — в `requirements.txt`,
`random_state` зафиксирован, повторный прогон даёт побайтово идентичный файл.

## 1. Окружение

In [1]:
import sys, platform, hashlib, warnings
import numpy as np
import pandas as pd
import sklearn
from scipy.stats import rankdata
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier

from metric import precision_at_recall

warnings.filterwarnings('ignore')
SEED = 42

print('Python      :', sys.version.split()[0], '|', platform.platform())
print('numpy       :', np.__version__)
print('pandas      :', pd.__version__)
print('scikit-learn:', sklearn.__version__)

Python      : 3.13.15 | Windows-10-10.0.19045-SP0
numpy       : 2.4.4
pandas      : 3.0.2
scikit-learn: 1.8.0


## 2. Загрузка и первый взгляд

In [2]:
train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test  = pd.read_csv('data/test.csv',  parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
raw   = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

y = train.target.values
print('train', train.shape, '| test', test.shape, '| events', raw.shape)
print('доля ботов:', round(y.mean(), 4))
print('окна train:', train.window_start_ts.min().date(), '-', train.window_start_ts.max().date())
print('окна test :', test.window_start_ts.min().date(), '-', test.window_start_ts.max().date())

train (11091, 5) | test (4909, 4) | events (328905, 14)
доля ботов: 0.0811
окна train: 2026-04-06 - 2026-04-19
окна test : 2026-04-20 - 2026-04-26


Окна теста идут строго после окон трейна. Отсюда первое решение: валидация только временна́я.
Случайный сплит дал бы оптимистичную и бесполезную оценку, потому что модель увидела бы будущее.

In [3]:
print(raw.event_name.value_counts(), '\n')
print(raw.platform.value_counts(), '\n')
print('доля пропусков:'); print(raw.isna().mean().round(3), '\n')
print('полных дублей строк:', raw.duplicated().sum())
print('события отсортированы по времени внутри куки:',
      raw.groupby('cookie_id').event_ts.is_monotonic_increasing.all())

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64 

доля пропусков:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
dtype: float64 

полных дублей строк: 4863
события отсортированы по времени внутри куки: 

Три вещи, которые определяют шаг очистки.

**`platform` записана в разном регистре**: `WEB` / `Web` / `web`, `ANDROID` / `Android` / `android`,
`IOS` / `iOS` / `ios` / `iphone`. Без нормализации получится категориальная переменная с одиннадцатью
уровнями вместо четырёх.

**Пропуски структурные, а не случайные.** `search_query` и `search_page` пусты там, где событие не
является поиском, `pointer_x/y` — там, где курсора не существует (мобильные платформы). Это не
«данные потеряли», а «такого поля у этого события не бывает». Заполнять медианой нельзя: для
признака вида «разброс координат курсора» пропуск означает «курсора не было вовсе», и если забить
его нулём, это слипнется с «курсор был и не двигался» — а это разные вещи. Поэтому пропуски
оставляю как есть, `HistGradientBoostingClassifier` обрабатывает их нативно, отправляя в ту ветку,
которая лучше делит выборку.

**Дубли и порядок строк.** 4863 полностью одинаковые строки — артефакт выгрузки, удаляю: иначе
поедут все счётчики и интервалы между событиями. Файл не отсортирован, а без сортировки по
`(cookie_id, event_ts)` любой `diff()` по времени даст мусор.

In [4]:
events = raw.drop_duplicates().copy()
events['platform'] = (events.platform.astype('string').str.strip().str.lower()
                      .replace({'iphone': 'ios'}))
events = events.sort_values(['cookie_id', 'event_ts'], kind='mergesort').reset_index(drop=True)

print(len(raw), '->', len(events), f'(удалено {len(raw) - len(events)})')
print(sorted(events.platform.unique()))

328905 -> 324042 (удалено 4863)
['android', 'desktop', 'ios', 'web']


## 3. Окно наблюдения

Признаки считаются только по событиям внутри окна — требование условия, а не выбор. Левая граница
включается, правая нет.

In [5]:
def in_window(ev, meta):
    e = ev.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    e = e[(e.event_ts >= e.window_start_ts) & (e.event_ts < e.window_end_ts)]
    return e.sort_values(['cookie_id', 'event_ts'], kind='mergesort').reset_index(drop=True)

ev_tr = in_window(events, train)
ev_te = in_window(events, test)
all_ev = pd.concat([ev_tr, ev_te], ignore_index=True)

print('событий в окнах:', len(ev_tr), '/', len(ev_te))
print('куки без событий:', train.cookie_id.nunique() - ev_tr.cookie_id.nunique(),
      '/', test.cookie_id.nunique() - ev_te.cookie_id.nunique())

событий в окнах: 195484 / 88349
куки без событий: 0 / 0


## 4. Схема валидации

Пять расширяющихся окон: валидация на двух днях, обучение на всём, что строго раньше. Первый фолд
всегда слабый — обучение идёт на четырёх днях, — но это нехватка данных, а не дефект модели: у
финальной модели весь трейн.

Сначала я мерил на двух фолдах и это оказалось ошибкой. На двух фолдах суммарно около 260 ботов, и
разброс между соседними конфигурациями сопоставим с шумом, то есть решения принимались по случайным
колебаниям. Пять фолдов дают вдвое более устойчивую оценку.

In [6]:
FOLDS = [('2026-04-11', '2026-04-13'), ('2026-04-13', '2026-04-15'), ('2026-04-15', '2026-04-17'),
         ('2026-04-17', '2026-04-19'), ('2026-04-19', '2026-04-20')]

PARAMS = dict(max_iter=400, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20,
              l2_regularization=1.0, early_stopping=False, random_state=SEED)

WS = train.window_start_ts.values

def folds():
    for a, b in FOLDS:
        yield (WS >= np.datetime64(a)) & (WS < np.datetime64(b)), WS < np.datetime64(a)

def cv(X, cols, params=PARAMS, ret=False):
    """Средний P@R0.7 по временным фолдам для одной модели."""
    res = []
    for va, tr in folds():
        m = HistGradientBoostingClassifier(**params).fit(X.loc[tr, cols], y[tr])
        res.append(precision_at_recall(y[va], m.predict_proba(X.loc[va, cols])[:, 1]))
    return (round(float(np.mean(res)), 4), [round(r, 4) for r in res]) if ret else float(np.mean(res))

## 5. Baseline

Два счётчика и случайный лес, как в quickstart. Нужен только как точка отсчёта.

In [7]:
g = ev_tr.groupby('cookie_id')
base = (train[['cookie_id']]
        .merge(pd.DataFrame({'n_events': g.size(), 'u_item': g.item_id.nunique()}).reset_index(),
               on='cookie_id', how='left').fillna(0))

res = []
for va, tr in folds():
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=SEED)
    rf.fit(base.loc[tr, ['n_events', 'u_item']], y[tr])
    res.append(precision_at_recall(y[va], rf.predict_proba(base.loc[va, ['n_events', 'u_item']])[:, 1]))

print('baseline P@R0.7:', round(np.mean(res), 4), '| константа:', round(y.mean(), 4))

baseline P@R0.7: 0.0946 | константа: 0.0811


Baseline практически неотличим от константы. Объём активности сам по себе бота не выдаёт, и это
логично: человек, который полдня листает объявления, тоже генерирует много событий. Копать надо не в
«сколько», а в «как».

## 6. Признаки

Каждая группа — отдельная функция, принимает события внутри окна и возвращает таблицу с индексом
`cookie_id`. Так их удобно подключать по одной и мерить эффект.

In [8]:
EVENTS = ['item_view', 'search_results_view', 'photo_swipe', 'favorite_add', 'seller_page_view',
          'contact_phone_show', 'captcha_shown', 'login', 'contact_chat_open', 'contact_message_sent']

def entropy(counts):
    c = np.asarray(counts, float); c = c[c > 0]
    if c.sum() == 0:
        return 0.0
    p = c / c.sum()
    return float(-(p * np.log(p)).sum())

def counts_table(ev):
    return pd.crosstab(ev.cookie_id, ev.event_name).reindex(columns=EVENTS, fill_value=0)

### 6.1 Типы событий

Счётчики и доли. Доли важнее: они не зависят от общего объёма активности. Плюс энтропия по типам —
насколько равномерно размазана активность.

In [9]:
def f_counts(ev):
    ct = counts_table(ev)
    n = ev.groupby('cookie_id').size()
    f = pd.DataFrame(index=ct.index)
    for c in EVENTS:
        f['cnt_' + c] = ct[c]
        f['shr_' + c] = ct[c] / n
    f['n_events'] = n
    f['n_event_types'] = (ct > 0).sum(axis=1)
    f['event_type_entropy'] = ct.apply(lambda r: entropy(r.values), axis=1)
    return f

### 6.2 Воронка

Парсер много смотрит и почти не взаимодействует, человек наоборот: посмотрел пять объявлений, в два
написал. Считаю отношения к числу просмотров, а не абсолюты — они устойчивы к разному уровню
активности.

In [10]:
def f_ratios(ev):
    ct = counts_table(ev)
    v = ct.item_view.clip(lower=1)
    f = pd.DataFrame(index=ct.index)
    f['r_phone_per_view']  = ct.contact_phone_show / v
    f['r_swipe_per_view']  = ct.photo_swipe / v
    f['r_fav_per_view']    = ct.favorite_add / v
    f['r_seller_per_view'] = ct.seller_page_view / v
    f['r_contact_any']     = (ct.contact_chat_open + ct.contact_message_sent + ct.contact_phone_show) / v
    f['r_search_per_view'] = ct.search_results_view / v
    f['has_login']   = (ct.login > 0).astype(int)
    f['has_captcha'] = (ct.captcha_shown > 0).astype(int)
    return f

### 6.3 Разнообразие

Что именно смотрит кука. Парсер обходит каталог вширь: много категорий, много городов, мало
повторов. Человек ищет конкретную вещь и возвращается к одним и тем же объявлениям.

In [11]:
def f_diversity(ev):
    g = ev.groupby('cookie_id'); ct = counts_table(ev)
    f = pd.DataFrame(index=g.size().index)
    f['u_item']        = g.item_id.nunique()
    f['u_category']    = g.item_category.nunique()
    f['u_location']    = g.item_location.nunique()
    f['u_query']       = g.search_query.nunique()
    f['u_seller_type'] = g.seller_type.nunique()
    f['item_repeat_rate'] = 1 - f.u_item / ct.item_view.clip(lower=1)
    f['cat_per_item']     = f.u_category / f.u_item.clip(lower=1)   # "ширина" обхода
    f['loc_per_item']     = f.u_location / f.u_item.clip(lower=1)
    f['query_per_search'] = f.u_query / ct.search_results_view.clip(lower=1)
    return f

### 6.4 Пагинация

Человек редко уходит дальше второй страницы выдачи, парсер идёт по страницам подряд.
`page_seq_share` — доля переходов ровно на +1 внутри одного запроса.

In [12]:
def f_pages(ev):
    sp = ev.dropna(subset=['search_page'])
    f = pd.DataFrame(index=pd.Index(sorted(ev.cookie_id.unique()), name='cookie_id'))
    if len(sp) == 0:
        for c in ['page_max', 'page_mean', 'page_deep_share', 'page_seq_share']:
            f[c] = np.nan
        return f
    g = sp.groupby('cookie_id')
    f['page_max']  = g.search_page.max()
    f['page_mean'] = g.search_page.mean()
    f['page_deep_share'] = g.search_page.apply(lambda x: (x > 1).mean())
    sp2 = sp.sort_values(['cookie_id', 'search_query', 'event_ts'])
    step = sp2.groupby(['cookie_id', 'search_query']).search_page.diff()
    f['page_seq_share'] = (step == 1).groupby(sp2.cookie_id).mean()
    return f

### 6.5 Время

Главная содержательная группа. У скрипта с фиксированной паузой интервалы между событиями
кучкуются, у человека размазаны: то очередь событий за секунду, то пауза на полчаса.

Отдельно стоит берстность `(σ−μ)/(σ+μ)`. У пуассоновского потока, где события независимы, σ ≈ μ и
индекс около нуля; у строго регулярного σ → 0 и индекс → −1; у пачечного → +1. Одно безразмерное
число вместо трёх коррелирующих статистик, и сравнивать можно куки с разной активностью.

In [13]:
def f_time(ev):
    ev = ev.assign(_t=ev.event_ts.astype('int64') // 10**9)
    g = ev.groupby('cookie_id')
    dt = g._t.diff()
    dg = dt.groupby(ev.cookie_id)
    f = pd.DataFrame(index=g.size().index)
    f['dt_mean']   = dg.mean()
    f['dt_median'] = dg.median()
    f['dt_std']    = dg.std()
    f['dt_min']    = dg.min()
    f['dt_max']    = dg.max()
    f['dt_cv']     = f.dt_std / f.dt_mean.replace(0, np.nan)
    f['burstiness'] = (f.dt_std - f.dt_mean) / (f.dt_std + f.dt_mean).replace(0, np.nan)
    f['dt_fast_share'] = dg.apply(lambda x: (x < 1).mean())
    f['dt_lt3_share']  = dg.apply(lambda x: (x < 3).mean())
    # доля самого частого интервала с округлением до секунды — прямая мера "тикания по таймеру"
    f['dt_mode_share'] = dg.apply(lambda x: x.round().value_counts(normalize=True).max()
                                  if x.notna().any() else np.nan)
    f['span_sec'] = g._t.max() - g._t.min()
    f['rate_per_hour'] = g.size() / (f.span_sec / 3600).replace(0, np.nan)
    f['n_sessions'] = (dt.isna() | (dt > 1800)).astype(int).groupby(ev.cookie_id).sum()
    f['events_per_session'] = g.size() / f.n_sessions.clip(lower=1)
    hour = ev.event_ts.dt.hour
    f['hours_active'] = hour.groupby(ev.cookie_id).nunique()
    f['hour_entropy'] = hour.groupby(ev.cookie_id).apply(lambda x: entropy(x.value_counts().values))
    f['night_share']  = hour.groupby(ev.cookie_id).apply(lambda x: x.isin(range(6)).mean())
    return f

### 6.6 Платформа и user-agent

Сырой UA как категорию брать нельзя: в тесте полезут строки, которых не было в трейне, и модель про
них ничего не знает. Разбираю на устойчивые куски: какая ОС, какая мажорная версия, нет ли явных
маркеров автоматизации. Плюс две идеи — сколько разных UA у куки за сутки (ротация это аномалия) и
согласуется ли UA с заявленной платформой.

In [14]:
def parse_ua(s):
    txt = s.fillna('').astype(str); low = txt.str.lower()
    out = pd.DataFrame(index=s.index)
    out['len']      = txt.str.len()
    out['android']  = low.str.contains('android').astype(int)
    out['ios']      = low.str.contains('iphone|ipad|ios').astype(int)
    out['windows']  = low.str.contains('windows').astype(int)
    out['mac']      = low.str.contains('macintosh|mac os').astype(int)
    out['linux']    = (low.str.contains('linux') & ~low.str.contains('android')).astype(int)
    out['headless'] = low.str.contains('headless|phantom|electron').astype(int)
    out['lib']      = low.str.contains('curl|python|requests|go-http|okhttp|scrapy|java/|wget').astype(int)
    out['botword']  = low.str.contains('bot|spider|crawler').astype(int)
    out['major_ver'] = pd.to_numeric(
        txt.str.extract(r'(?:Chrome|Firefox|Version)/(\d+)', expand=False), errors='coerce')
    return out

def f_platform_ua(ev):
    g = ev.groupby('cookie_id'); n = g.size()
    f = pd.DataFrame(index=n.index)
    f['u_platform']   = g.platform.nunique()
    f['u_user_agent'] = g.user_agent.nunique()
    pf = pd.crosstab(ev.cookie_id, ev.platform)
    for p in ['desktop', 'web', 'android', 'ios']:
        f['plat_shr_' + p] = (pf[p] / n) if p in pf.columns else 0.0
    ua = parse_ua(ev.user_agent); ua['cookie_id'] = ev.cookie_id.values
    f = f.join(ua.groupby('cookie_id').mean(numeric_only=True).add_prefix('ua_'))
    f['ua_plat_mismatch'] = ((f.ua_android - f.plat_shr_android).abs()
                             + (f.ua_ios - f.plat_shr_ios).abs()) / 2
    return f

### 6.7 Курсор

Координаты есть только у части событий, поэтому отдельно считаю заполненность, отдельно статистики
по непустым. Эмулятор переиспользует одни и те же точки, живая рука так не умеет.

In [15]:
def f_pointer(ev):
    g = ev.groupby('cookie_id')
    f = pd.DataFrame(index=g.size().index)
    f['pointer_fill'] = g.pointer_x.apply(lambda x: x.notna().mean())
    px = ev.dropna(subset=['pointer_x', 'pointer_y'])
    if len(px):
        gp = px.groupby('cookie_id')
        f['ptr_x_std'] = gp.pointer_x.std()
        f['ptr_y_std'] = gp.pointer_y.std()
        f['ptr_unique_share'] = gp.apply(lambda d: len(set(zip(d.pointer_x, d.pointer_y))) / len(d),
                                         include_groups=False)
    return f

### 6.8 Метаданные куки

Единственное, что есть помимо событий, — когда кука создана. Свежая кука с большой активностью
подозрительна.

День недели и час начала окна я пробовал и **выбросил**: они описывают нарезку данных, а не
поведение куки. Модель по ним частично запоминает, из какого дня пример, а в тесте дни другие.
Проверка в разделе 9.

In [16]:
def f_meta(meta):
    m = meta.set_index('cookie_id')
    age = (m.window_start_ts - m.cookie_created_at).dt.total_seconds() / 86400
    return pd.DataFrame({'cookie_age_days': age, 'cookie_fresh': (age < 1).astype(int)}, index=m.index)

### 6.9 Концентрация обхода

`u_category` считает, сколько разных категорий смотрела кука. Но важнее не количество, а
**равномерность**: у парсера обход ровный, у человека один явный фокус и хвост случайных заходов.
Индекс Херфиндаля `Σp²` это ловит: 1 — всё в одной категории, ближе к нулю — размазано.

In [17]:
def hhi(x):
    v = x.dropna()
    if len(v) == 0:
        return np.nan
    p = v.value_counts(normalize=True).values
    return float((p ** 2).sum())

def f_conc(ev):
    g = ev.groupby('cookie_id')
    return pd.DataFrame({
        'hhi_category':  g.item_category.apply(hhi),
        'hhi_location':  g.item_location.apply(hhi),
        'hhi_query':     g.search_query.apply(hhi),
        'items_per_cat': g.item_id.nunique() / g.item_category.nunique().clip(lower=1),
    })

### 6.10 Пиковая интенсивность

Средний темп у меня уже есть (`rate_per_hour`), но показателен не средний темп, а **пик**: всплеск в
десятки событий за минуту человеку физически недоступен, а скрипту достаётся даром. Отношение пика к
медиане нормирует это на общий уровень активности.

In [18]:
def f_peak(ev):
    g = ev.groupby('cookie_id')
    f = pd.DataFrame(index=g.size().index)
    per_min = ev.assign(_m=ev.event_ts.dt.floor('1min')).groupby(['cookie_id', '_m']).size().groupby('cookie_id')
    f['max_ev_per_min'] = per_min.max()
    f['med_ev_per_min'] = per_min.median()
    f['peak_to_med_1m'] = f.max_ev_per_min / f.med_ev_per_min.replace(0, np.nan)
    per_5m = ev.assign(_m=ev.event_ts.dt.floor('5min')).groupby(['cookie_id', '_m']).size().groupby('cookie_id')
    f['max_ev_per_5min'] = per_5m.max()
    f['peak_to_med_5m']  = per_5m.max() / per_5m.median().replace(0, np.nan)
    f['active_minutes']  = ev.assign(_m=ev.event_ts.dt.floor('1min')).groupby('cookie_id')._m.nunique()
    return f

### 6.11 Контекст по корпусу

Все признаки выше описывают куку изолированно. Но есть и контекст: насколько популярны объявления,
которые она смотрит, и сколько других куки сидят на том же user-agent. Логика: парсер обходит
массово и попадает на те же позиции, что и другие парсеры, человек смотрит свою частную выборку.

Меток здесь нет, только события, поэтому утечки таргета не возникает. С временем сложнее — разбираю
это отдельно в разделе 8, там главный сюжет.

In [19]:
def ua_norm(s):
    """Схлопываем версии в 'N': Chrome/119.0.0.0 и Chrome/120.0.0.0 — один агент."""
    return s.fillna('').astype(str).str.lower().str.replace(r'\d+(\.\d+)*', 'N', regex=True)

def f_cross(ev_src, idx):
    """Контекст по корпусу. ev_src задаёт, по каким событиям он считается, — ключевой параметр."""
    e = ev_src.assign(_ua=ua_norm(ev_src.user_agent))
    ck_ua = e.groupby('cookie_id')._ua.agg(lambda s: s.mode().iat[0] if len(s) else '')
    out = pd.DataFrame(index=idx)
    out['ua_share_count'] = ck_ua.map(ck_ua.value_counts()).reindex(idx)

    iv = e[e.event_name == 'item_view']
    cip = iv.assign(p=iv.item_id.map(iv.groupby('item_id').cookie_id.nunique())).groupby('cookie_id').p
    out['item_pop_mean'] = cip.mean().reindex(idx)
    out['item_pop_max']  = cip.max().reindex(idx)

    sq = e.dropna(subset=['search_query'])
    cqp = sq.assign(p=sq.search_query.map(sq.groupby('search_query').cookie_id.nunique())).groupby('cookie_id').p
    out['query_pop_mean'] = cqp.mean().reindex(idx)

    lv = e.dropna(subset=['item_location'])
    clp = lv.assign(p=lv.item_location.map(lv.groupby('item_location').cookie_id.nunique())).groupby('cookie_id').p
    out['loc_pop_mean'] = clp.mean().reindex(idx)
    return out

### Сборка

In [20]:
%%time
def build(ev, meta):
    parts = [f_counts(ev), f_ratios(ev), f_diversity(ev), f_pages(ev),
             f_time(ev), f_platform_ua(ev), f_pointer(ev)]
    X = pd.concat(parts, axis=1).join(f_meta(meta), how='right')
    X['n_events'] = X.n_events.fillna(0)          # куки без событий: активности честно ноль
    return X.reindex(meta.set_index('cookie_id').index)

X_tr = build(ev_tr, train)
X_te = build(ev_te, test)
CORE = list(X_tr.columns)

for name, ftr, fte in [('conc', f_conc(ev_tr), f_conc(ev_te)),
                       ('peak', f_peak(ev_tr), f_peak(ev_te)),
                       ('cross', f_cross(all_ev, X_tr.index).add_prefix('cr_'),
                                 f_cross(all_ev, X_te.index).add_prefix('cr_'))]:
    X_tr = X_tr.join(ftr.reindex(X_tr.index))
    X_te = X_te.join(fte.reindex(X_te.index))
    globals()[name.upper()] = list(ftr.columns)

X_tr, X_te = X_tr.reset_index(), X_te.reset_index()
FEATURES = CORE + CONC + PEAK + CROSS

assert list(X_tr.cookie_id) == list(train.cookie_id)
assert list(X_te.cookie_id) == list(test.cookie_id)
print(len(CORE), 'базовых +', len(CONC), 'концентрация +', len(PEAK), 'пик +',
      len(CROSS), 'контекст =', len(FEATURES))

84 базовых + 4 концентрация + 6 пик + 5 контекст = 99
CPU times: total: 40.2 s
Wall time: 40.5 s


## 7. Замер групп по очереди

Добавляю группу, смотрю на среднее по пяти фолдам.

In [21]:
%%time
GROUPS = [('счётчики', list(f_counts(ev_tr).columns)),
          ('воронка', list(f_ratios(ev_tr).columns)),
          ('разнообразие', list(f_diversity(ev_tr).columns)),
          ('пагинация', list(f_pages(ev_tr).columns)),
          ('время', list(f_time(ev_tr).columns)),
          ('платформа+UA', list(f_platform_ua(ev_tr).columns)),
          ('курсор', list(f_pointer(ev_tr).columns)),
          ('мета куки', ['cookie_age_days', 'cookie_fresh']),
          ('концентрация', CONC),
          ('пик', PEAK),
          ('контекст', CROSS)]

cols = []
for name, gcols in GROUPS:
    cols = [c for c in FEATURES if c in set(cols) | set(gcols)]
    print(f'{name:14s} {len(cols):3d} признаков -> {cv(X_tr, cols):.4f}')

счётчики        23 признаков -> 0.1537
воронка         31 признаков -> 0.1562
разнообразие    40 признаков -> 0.2137
пагинация       44 признаков -> 0.2627
время           61 признаков -> 0.3180
платформа+UA    78 признаков -> 0.3409
курсор          82 признаков -> 0.4920
мета куки       84 признаков -> 0.5399
концентрация    88 признаков -> 0.5835
пик             94 признаков -> 0.6501
контекст        99 признаков -> 0.7597
CPU times: total: 9min
Wall time: 2min 24s


Три наблюдения.

**Разнообразие даёт первый настоящий скачок.** То, *что* кука смотрит, информативнее того,
*сколько*.

**Курсор даёт самый большой прирост из «внутренних» групп** — это стоит проверить отдельно, слишком
красиво.

**Контекст по корпусу даёт ещё больше** — и это стоит проверить особенно тщательно.

In [22]:
%%time
ptr = list(f_pointer(ev_tr).columns)
print('только курсор  :', round(cv(X_tr, ptr), 4))
print('всё без курсора:', round(cv(X_tr, [c for c in FEATURES if c not in ptr]), 4))
print('всё            :', round(cv(X_tr, FEATURES), 4))

только курсор  : 0.1125
всё без курсора: 0.6695
всё            : 0.7597
CPU times: total: 2min 35s
Wall time: 40.4 s


Сам по себе курсор почти ничего не даёт, но без него полный набор заметно падает. Значит это не
признак-читер, а **взаимодействие**: разброс координат разделяет куки только в сочетании с профилем
активности. Отсюда практический вывод: permutation importance на этих данных врёт — переставляешь
один признак, а информацию несут соседи, и важность выходит заниженной. Считать надо группами, что я
и делаю.

## 8. Проверка контекста на утечку во времени

Подозрение. `item_pop_mean` для куки из 7 апреля считается по всем событиям корпуса, включая конец
апреля. То есть признак смотрит в будущее относительно окна наблюдения этой куки, а условие требует,
чтобы признаки были доступны на момент `window_end_ts`.

Проверяю в лоб: пересчитываю контекст для каждого фолда, используя только события, случившиеся **до
начала валидационного окна**.

In [23]:
%%time
inner = CORE + CONC + PEAK
rows = []
for mode in ['без контекста', 'контекст по всему корпусу', 'контекст только из прошлого']:
    res = []
    for (va, tr), (a, b) in zip(folds(), FOLDS):
        if mode == 'без контекста':
            Xf, cols = X_tr, inner
        else:
            src = all_ev if mode == 'контекст по всему корпусу' else all_ev[all_ev.event_ts < np.datetime64(a)]
            cr = f_cross(src, pd.Index(train.cookie_id)).add_prefix('cr_').reset_index(drop=True)
            Xf = pd.concat([X_tr[inner], cr], axis=1); cols = inner + CROSS
        m = HistGradientBoostingClassifier(**PARAMS).fit(Xf.loc[tr, cols], y[tr])
        res.append(precision_at_recall(y[va], m.predict_proba(Xf.loc[va, cols])[:, 1]))
    rows.append({'вариант': mode, 'P@R0.7': round(np.mean(res), 4), 'по фолдам': [round(x, 4) for x in res]})

pd.DataFrame(rows)

CPU times: total: 3min 32s
Wall time: 1min 7s


,вариант,P@R0.7,по фолдам
0,без контекста,0.6501,"[0.5337, 0.6127, 0.5871, 0.7477, 0.7692]"
1,контекст по всему корпусу,0.7597,"[0.5598, 0.8529, 0.8318, 0.7434, 0.8108]"
2,контекст только из прошлого,0.6081,"[0.4703, 0.6286, 0.6154, 0.6288, 0.6977]"


Числа: без контекста 0.650, по всему корпусу 0.760, только из прошлого 0.608.

Первое прочтение — «весь прирост держится на заглядывании вперёд, признак надо выбросить». Я так и
решил, и **это было ошибкой**, которую показала отправка на лидерборд: вариант с контекстом дал там
0.835 против 0.726 без него, то есть прирост перенёсся полностью.

Где ошибка в рассуждении. Ограничивая контекст прошлым, я убирал не только заглядывание вперёд, но и
**объём данных, по которым оценивается популярность**. На ранних фолдах это считанные дни корпуса,
оценки становятся шумными, и признак деградирует по совершенно другой причине.

Содержательно правильная рамка: популярность объявления — свойство корпуса, а не куки. События всех
тестовых куки нам выданы, значит для теста популярность оценивается по тому же полному пулу, что и
для трейна. Признак симметричен между обучением и применением, меток не использует, а валидационный
фолд с полным контекстом воспроизводит ситуацию теста точнее, чем причинный вариант. Формально
ограничение про `window_end_ts` он натягивает — это честное ограничение решения, — но по существу
работает.

Урок общий: причинная проверка признака должна менять **только** причинность, а не объём данных
заодно. Иначе результат неинтерпретируем.

## 9. Что проверено и отброшено

Из примерно семидесяти проверенных гипотез рабочими оказались пять. Здесь — самое поучительное из
отброшенного, с числами.

In [24]:
%%time
# Target encoding user-agent (fold-safe: кодировка считается только по train-части фолда)
def te_fit(keys, yy, prior, sm):
    d = pd.DataFrame({'k': keys, 'y': yy}).groupby('k').y.agg(['sum', 'count'])
    return (d['sum'] + prior * sm) / (d['count'] + sm)

ua_key = ua_norm(all_ev.user_agent).groupby(all_ev.cookie_id).agg(lambda s: s.mode().iat[0])
keys = ua_key.reindex(train.cookie_id).fillna('').values
print('различных нормализованных UA во всём корпусе:', ua_key.nunique())

res = []
for va, tr in folds():
    pri = y[tr].mean()
    enc = te_fit(keys[tr], y[tr], pri, 30)
    Xf = X_tr[FEATURES].copy()
    Xf['te_ua'] = pd.Series(keys).map(enc).fillna(pri).values
    m = HistGradientBoostingClassifier(**PARAMS).fit(Xf.loc[tr], y[tr])
    res.append(precision_at_recall(y[va], m.predict_proba(Xf.loc[va])[:, 1]))
print('+ TE user_agent:', round(np.mean(res), 4))

различных нормализованных UA во всём корпусе: 19
+ TE user_agent: 0.7677
CPU times: total: 1min 7s
Wall time: 18.7 s


Одиночная модель показывает крошечный плюс, но это ровно тот масштаб, который на пяти фолдах
неотличим от шума, а структурный аргумент против — в числе различных агентов: после схлопывания
версий их во всём корпусе меньше двух десятков. Это не высококардинальная категория, ради которой
target encoding и придумывают, а обычный фактор на 19 уровней, который модель и так достаёт из
имеющихся `ua_*` признаков. Признак, способный лишь переформулировать уже имеющуюся информацию,
добавляет дисперсии, не добавляя сигнала. Та же картина с target encoding дня недели.

Календарные признаки (`window_dow`, `window_hour_start`) я проверяю ниже, в разделе 10, вместе с
бэггингом — там видно, почему одиночная модель по таким вопросам ненадёжный судья.

In [25]:
%%time
# Отбор признаков по важности (важности считаются только на train-части фолда)
for k in [30, 45]:
    res = []
    for va, tr in folds():
        rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=SEED)
        rf.fit(X_tr.loc[tr, FEATURES].fillna(-999), y[tr])
        top = list(pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False).head(k).index)
        m = HistGradientBoostingClassifier(**PARAMS).fit(X_tr.loc[tr, top], y[tr])
        res.append(precision_at_recall(y[va], m.predict_proba(X_tr.loc[va, top])[:, 1]))
    print(f'топ-{k}: {np.mean(res):.4f}')
print(f'все {len(FEATURES)}: {cv(X_tr, FEATURES):.4f}')

топ-30: 0.7584
топ-45: 0.7426
все 99: 0.7597
CPU times: total: 3min 56s
Wall time: 50.7 s


Обрезать набор нельзя: чем меньше признаков, тем хуже. Слабые поодиночке работают в связке — ровно то
же, что показала проверка курсора.

**Вырожденный признак.** Стоит посмотреть на `n_sessions`: у всех куки он равен единице. Разрыв
больше 30 минут внутри суточного окна не встречается ни разу, то есть `n_sessions` — константа, а
`events_per_session` просто дублирует `n_events`. Целая идея с сессиями на этих данных не работает,
хотя выглядела разумной.

In [26]:
print(X_tr.n_sessions.value_counts())

n_sessions
1    11091
Name: count, dtype: int64


**Что ещё проверено тем же способом и не дало прироста** (все в пределах ±0.02 от базы или хуже):

* квантили и скошенность интервалов, автокорреляция интервалов на лаге 1, энтропия округлённых
  интервалов, тренд темпа по ходу окна;
* временна́я концентрация: HHI по минутным и часовым корзинам, доля событий в самой загруженной
  минуте и часе, максимум за 10 секунд, длина и число пачек событий с интервалом до 2 секунд,
  коэффициент Джини по интервалам — всё семейство легло в 0.647-0.658 против 0.650, потому что
  полезный сигнал уже вытащен признаком `peak_to_med_1m`;
* энтропия переходов между типами событий (0.604), доля просмотров без предшествующего поиска
  (0.608), задержка от показа выдачи до перехода в объявление;
* динамика курсора: скорость, ускорение, извилистость траектории;
* текстовые свойства запросов: длина, число слов, доля с цифрами, переиспользование;
* новизна и синхронность объявлений (возраст объявления на момент просмотра, доля первых
  просмотров, число куки на том же объявлении в тот же час);
* сырой user_agent категорией, логарифмы объёмных признаков (деревья инвариантны к монотонным
  преобразованиям — число не меняется вообще), число дублей строк у куки, статистики длин сессий;
* перебор гиперпараметров: семь конфигураций, разброс ±0.05 — чистый шум.

Характерная ловушка: несколько из этих групп по отдельности как будто давали +0.02...+0.04, а все
вместе оказывались хуже исходного набора. Это диагностический признак того, что приросты лежали в
шуме. Под бэггингом, который дисперсию режет, они исчезли окончательно.

## 10. Ансамбль: бэггинг и свёртка рангов

Метрика ранжирующая и шумная, значит проблема в дисперсии, а не в смещении, — и бить надо именно по
ней. Обучаю модели на бутстрэп-выборках и усредняю **ранги**, а не вероятности: шкалы моделей
несопоставимы, а для ранжирующей метрики важен только порядок.

Дальше вопрос, чем свернуть ранги. Сравнение здесь **парное**: модели те же, данные те же,
различается только способ свёртки, поэтому вся дисперсия от выбора фолда и обучения сокращается, и
различия в 0.01 уже осмысленны.

In [27]:
%%time
def bagged(cols, va, tr, n_bag=8, seed=SEED):
    """Возвращает матрицу вероятностей (n_bag x n_val) для одного фолда."""
    Xa, ya, Xv = X_tr.loc[tr, cols], y[tr], X_tr.loc[va, cols]
    rng = np.random.default_rng(seed)
    out = []
    for i in range(n_bag):
        idx = rng.choice(len(Xa), len(Xa), replace=True)
        m = HistGradientBoostingClassifier(**{**PARAMS, 'random_state': 100 + i})
        out.append(m.fit(Xa.iloc[idx], ya[idx]).predict_proba(Xv)[:, 1])
    return np.array(out)

# Календарь проверяю здесь же: одиночная модель по таким вопросам ненадёжна.
CAL = ['window_dow', 'window_hour_start']
X_tr[CAL[0]] = train.window_start_ts.dt.dayofweek.values
X_tr[CAL[1]] = train.window_start_ts.dt.hour.values

rows = []
for name, cols in [('финальный набор', FEATURES), ('+ календарь', FEATURES + CAL)]:
    single, mean_r, med_r = [], [], []
    for (va, tr) in folds():
        P = bagged(cols, va, tr)
        R = np.array([rankdata(p) for p in P])
        mean_r.append(precision_at_recall(y[va], R.mean(0)))
        med_r.append(precision_at_recall(y[va], np.median(R, 0)))
        m = HistGradientBoostingClassifier(**PARAMS).fit(X_tr.loc[tr, cols], y[tr])
        single.append(precision_at_recall(y[va], m.predict_proba(X_tr.loc[va, cols])[:, 1]))
    rows.append({'набор': name, 'одна модель': round(np.mean(single), 4),
                 'бэггинг, среднее рангов': round(np.mean(mean_r), 4),
                 'бэггинг, медиана рангов': round(np.mean(med_r), 4)})

X_tr = X_tr.drop(columns=CAL)
pd.DataFrame(rows)

CPU times: total: 18min 46s
Wall time: 4min 42s


,набор,одна модель,"бэггинг, среднее рангов","бэггинг, медиана рангов"
0,финальный набор,0.7597,0.8089,0.8269
1,+ календарь,0.7791,0.8146,0.8141


Два вывода из одной таблицы.

**Бэггинг даёт заметный прирост, а медиана рангов — ещё около 0.02 сверху.** Медиана выигрывает у
среднего на всех пяти фолдах без исключения.

Механизм понятен. Одна модель из бутстрэпа может засунуть конкретную куку в резко нетипичное место
рейтинга — просто потому, что в её выборке не оказалось похожих примеров. Среднее тянется за таким
выбросом пропорционально 1/n, медиана его игнорирует. Это та же причина, по которой медиана
устойчивее среднего к тяжёлым хвостам.

**Календарь: одиночная модель ошибается.** С календарными признаками одна модель выглядит лучше, а
под бэггингом с медианой — хуже. Это ровно тот случай, ради которого стоит снижать дисперсию перед
тем, как принимать решение: кажущийся прирост в 0.02 у одиночной модели был шумом, и я едва не
оставил признак, описывающий нарезку данных, а не поведение куки. Календарь выброшен.

Общее правило, которое я вывел из этой задачи: если разница между вариантами меньше 0.03, одиночная
модель на пяти фолдах её не различает. Либо снижай дисперсию бэггингом, либо делай сравнение парным
(одни и те же модели, разная свёртка), либо не делай вывод.

## 11. Финальная модель и `submission.csv`

24 модели на бутстрэп-выборках, свёртка медианой рангов. Обучение на всём трейне: фолды нужны были
только для выбора конфигурации.

Среднюю вероятность подмешиваю микровесом 0.001 — не ради качества, а чтобы развязать одинаковые
медианные ранги. Метрика режет группы равных `score` целиком: отмечаются либо все куки с таким
значением, либо ни одна, — и ничьи обходятся дорого. Случайный лес из quickstart, например, выдаёт
вероятности кратно `1/n_estimators`, и куки слипаются в группы по несколько сотен.

In [28]:
%%time
N_BAG = 24

def final_score(seed=SEED, n_bag=N_BAG):
    """Единственный путь вычисления score — используется и для сабмита, и для проверки."""
    rng = np.random.default_rng(seed)
    P = []
    for i in range(n_bag):
        idx = rng.choice(len(X_tr), len(X_tr), replace=True)
        m = HistGradientBoostingClassifier(**{**PARAMS, 'random_state': 100 + i})
        m.fit(X_tr[FEATURES].iloc[idx], y[idx])
        P.append(m.predict_proba(X_te[FEATURES])[:, 1])
    P = np.array(P)
    median_rank = np.median(np.array([rankdata(p) for p in P]), 0) / len(X_te)
    return 0.999 * median_rank + 0.001 * P.mean(0)

sub = pd.DataFrame({'cookie_id': X_te.cookie_id, 'score': final_score()})
sub.to_csv('submission.csv', index=False)
sub.head()

CPU times: total: 6min 11s
Wall time: 1min 34s


,cookie_id,score
0,ck_315fb710a0e371e7,0.085268
1,ck_a76ee3b3e3e522fd,0.919916
2,ck_94c9a4d382689e82,0.540305
3,ck_8eaf9509ad9462a0,0.158733
4,ck_9a88a5a989cb5bc6,0.605834


In [29]:
chk = pd.read_csv('submission.csv')
assert list(chk.columns) == ['cookie_id', 'score'],      'ровно две колонки в нужном порядке'
assert len(chk) == len(test),                            f'ожидалось {len(test)} строк'
assert chk.cookie_id.is_unique,                          'дубликаты cookie_id'
assert set(chk.cookie_id) == set(test.cookie_id),        'набор cookie_id не совпадает с test.csv'
assert chk.score.notna().all(),                          'пропуски в score'
assert chk.score.between(0, 1).all(),                    'score вне [0, 1]'

print('строк:', len(chk))
print('уникальных score:', chk.score.nunique(), f'({chk.score.nunique() / len(chk):.1%})')
print('score: min %.4f | median %.4f | max %.4f' % (chk.score.min(), chk.score.median(), chk.score.max()))
print('все проверки пройдены')

строк: 4909
уникальных score: 4909 (100.0%)
score: min 0.0031 | median 0.4956 | max 0.9998
все проверки пройдены


In [30]:
%%time
# Воспроизводимость: тот же код, тот же seed — файл должен совпасть побайтово.
pd.DataFrame({'cookie_id': X_te.cookie_id, 'score': final_score()}).to_csv('_check.csv', index=False)
h = [hashlib.md5(open(f, 'rb').read()).hexdigest() for f in ('submission.csv', '_check.csv')]
print(h[0], '|', h[1], '| идентичны:', h[0] == h[1])

import os; os.remove('_check.csv')

7381f8e3a066d4c5c90d6520efceee5b | 7381f8e3a066d4c5c90d6520efceee5b | идентичны: True
CPU times: total: 6min 7s
Wall time: 1min 32s


## 12. Итоги

| шаг | P@R0.7, 5 фолдов |
|---|---|
| константа | 0.094 |
| baseline из quickstart | 0.099 |
| базовые признаки без календаря | 0.540 |
| + концентрация обхода | 0.584 |
| + пиковая интенсивность | 0.650 |
| + контекст по корпусу | 0.760 |
| + бэггинг, среднее рангов | 0.809 |
| + медиана рангов вместо среднего | **0.827** |

На скрытом тесте: 0.726 без контекста, 0.835 с контекстом и средним рангов, **0.839** с медианой.
Валидация систематически занижает тест примерно на 0.03, и это объяснимо: финальная модель учится на
всех 11 тысячах строк, а любой фолд — на меньшем.

**Что сработало:** концентрация обхода каталога, пиковая интенсивность, контекст по корпусу, бэггинг
по бутстрэпу и медиана рангов. Курсор — но только в связке с остальными признаками.

**Чего нет в решении сознательно:** сырых идентификаторов и UA как категорий (в тесте появляются
значения, которых не было в трейне), target encoding (при 900 положительных примерах переобучается
быстрее, чем приносит пользу), календарных признаков.

## Ограничения

**Оценка шумная.** 94-135 ботов на фолд, разброс между близкими конфигурациями сопоставим с шумом.
Решения принимались только по разнице больше 0.02 и по всем фолдам сразу; там, где разница была
меньше, я использовал парные сравнения (одни и те же модели, разная свёртка).

**Дрейф во времени.** Метрика падает от ранних фолдов к поздним. Модель привязана к поведению
парсеров конкретного периода, и заметная часть признаков обходится добавлением случайных задержек и
ротацией user-agent. Решение требует регулярного переобучения.

**Контекст по корпусу натягивает ограничение задачи.** Признаки `cr_*` используют события всего
корпуса, включая более поздние относительно окна конкретной куки. Меток они не используют и на тесте
работают, но если требование «доступно на момент `window_end_ts`» трактовать строго, эту группу
надо убрать — решение без неё даёт 0.726.

**`captcha_shown` — не независимый сигнал**, а следствие срабатывания уже существующего антибота.
Признак оставлен (он внутри окна и формально допустим), но опираться на него при оценке способности
ловить новые источники автоматизации нельзя.

**Потолок задачи.** Отрицательный класс в разметке — трафик, *отнесённый* к человеческому,
положительный — трафик *известных* сервисов. Метки получены эвристикой, значит часть ботов заведомо
сидит в нулях, и модель не разделит выборку лучше, чем разделена сама разметка.